# EX01. [추가연습] 회귀 실전 문제 - 캘리포니아 집값 예측

> Titanic이 아닌 새로운 데이터로 회귀 파이프라인 전체(EDA→전처리→ML→DL)를 처음부터 끝까지 다시 연습합니다. 메인 노트북(05, 08번)에서 배운 내용을 **다른 도메인에 응용**하는 것이 이 노트북의 목표입니다.

### 사용 데이터
`data/california_housing.csv` - 캘리포니아 각 지역구의 특성(소득, 방 개수, 위치 등)으로 **주택 가격(MedHouseVal)**을 예측합니다.

## 목차
1. 데이터 로딩 & EDA
2. 전처리
3. 머신러닝 회귀 모델링
4. 딥러닝 회귀 모델링
5. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

housing = pd.read_csv('../data/california_housing.csv')
housing.head()

## 1. 데이터 로딩 & EDA

### 📖 이론 설명
이 데이터는 전부 연속형(수치형) 변수로만 이루어져 있어 Titanic과는 성격이 다릅니다. 결측치나 범주형 인코딩보다는 **변수 간 스케일 차이**와 **분포의 치우침(skew)**을 먼저 살펴봐야 합니다.

### 💻 예제 코드

In [ ]:
housing.info()
print(housing.describe())

sns.histplot(housing['MedHouseVal'], kde=True)
plt.title('House Value Distribution')
plt.show()

sns.heatmap(housing.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `MedInc`(소득)과 `MedHouseVal`(주택가격)의 상관계수를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(housing['MedInc'].corr(housing['MedHouseVal']))  # 소득이 높을수록 집값도 높은 경향이 있는지 상관계수로 먼저 확인
```

</details>

**문제 2.** `MedInc`과 `MedHouseVal`의 관계를 jointplot으로 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.jointplot(data=housing, x='MedInc', y='MedHouseVal', kind='reg')  # kind='reg'로 회귀선을 함께 그려 선형 관계를 시각적으로 확인
plt.show()
```

</details>

## 2. 전처리

### 📖 이론 설명
결측치가 없는 데이터이므로 스케일링과 분할에 집중합니다. 컬럼마다 범위가 크게 달라(예: `Population`은 수백~수만, `AveRooms`은 한 자리 수) 스케일링이 특히 중요합니다.

### 💻 예제 코드

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(housing.isnull().sum().sum())  # 결측치 없음 확인

X = housing.drop(columns=['MedHouseVal'])
y = housing['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train.shape

### ✍️ TODO: 실전 문제

**문제 1.** 스케일링 전/후 `MedInc` 컬럼의 표준편차를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(X_train['MedInc'].std())  # 스케일링 전 표준편차
print(X_train_scaled[:, list(X.columns).index('MedInc')].std())  # 스케일링 후에는 넘파이 배열이라 컬럼 위치(인덱스)를 찾아 선택해야 함
```

</details>

## 3. 머신러닝 회귀 모델링

### 📖 이론 설명
05번 노트북과 동일한 5단계 템플릿을 그대로 적용합니다.

### 💻 예제 코드

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
print('RandomForest R2:', r2_score(y_test, rf_pred))

xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)
print('XGBoost R2:', r2_score(y_test, xgb_pred))

### ✍️ TODO: 실전 문제

**문제 1.** `RandomForest` 모델의 feature_importances_를 확인해 가장 중요한 변수를 `top_feature`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
imp = pd.Series(rf.feature_importances_, index=X.columns)  # feature_importances_ 배열에 컬럼명을 인덱스로 붙여 어떤 값이 어떤 변수인지 알 수 있게 함
top_feature = imp.idxmax()  # 값이 가장 큰 위치(컬럼명)를 반환
print(top_feature)
```

</details>

## 4. 딥러닝 회귀 모델링

### 📖 이론 설명
08번 노트북과 동일하게, 출력층 1노드(활성화 없음), `loss='mse'`로 설계합니다.

### 💻 예제 코드

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(100)
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=64,
                     validation_data=(X_test_scaled, y_test), callbacks=[es], verbose=0)

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend(); plt.title('Loss Curve'); plt.show()

dl_pred = model.predict(X_test_scaled).flatten()
print('DL R2:', r2_score(y_test, dl_pred))

## 5. 종합 실전 문제

**회귀 미니 모의고사입니다.**

**문제 1.** `GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)`를 학습시키고 R2를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import GradientBoostingRegressor
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)  # learning_rate: 부스팅 각 단계의 반영 비율
gb.fit(X_train_scaled, y_train)
print(r2_score(y_test, gb.predict(X_test_scaled)))
```

</details>

**문제 2.** 머신러닝(RandomForest)과 딥러닝 모델 중 어느 쪽 R2가 더 높은지 비교하는 코드를 작성하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print('RandomForest R2:', r2_score(y_test, rf_pred))  # 같은 평가지표(R2)로 ML과 DL을 나란히 비교해야 공정한 비교가 됨
print('DL R2:', r2_score(y_test, dl_pred))
```

</details>